In [ ]:
import tensorflow as tf
import numpy as np
import json
import pandas as pd
import os
from ipywidgets import interact
import ipywidgets as widgets

from Bio import SeqIO
from tqdm import tqdm
from pathlib import Path

from tfr_to_hash import (
    load_tfrecord_to_numpy,
    onehot_to_seq,
    deserialize,
    write_fasta,
    get_coords_from_blast,
    extract_region
    get_all_mouse_sequences,
    get_hg38, get_human_record_ids, get_human_basenji_regions,
    get_mm10, get_mouse_record_ids, get_mouse_basenji_regions,
    get_sequences_for_record,
    expand_regions,
    get_region_from_npy_filename,
    MOUSE_REF_FOLDER, MOUSE_TFR_FOLDER,
    HUMAN_REF_FOLDER, HUMAN_TFR_FOLDER,
    chunk_by_subset
)

In [ ]:
# genome_fasta_files = [ f'data/datasets/ref/human/chr{chromosome}.fa' for chromosome in list(range(1,23)) + ["X", "Y", "MT"]]
# seqs_per_chr = { record.id: str(record.seq).lower() for record in tqdm(SeqIO.parse(genome_fasta, "fasta")) }

In [ ]:
mm10_per_chr   = get_mm10()
mouse_seqs_df  = get_mouse_basenji_regions()
mouse_seqs_set = set(mouse_seqs_df.apply(extract_region, axis=1).to_list())

In [ ]:
train_files, valid_files, test_files = get_tfr_folder(MOUSE_TFR_FOLDER)
train_files = train_files + valid_files

In [ ]:
metadata = dict(seq_length=131072, target_length=896, num_targets=1643)
records  = load_tfrecord_to_numpy(test_files[0], metadata=metadata)

In [ ]:
import h5py

with h5py.File("test_mouse_copy.h5", "r+") as f:
    dset_target = f["target"]
    num_samples = dset_target.shape[0]

    start = 0
    for i, test_file in enumerate(test_files):
    # for start in tqdm(range(0, num_samples, chunk_size), desc="Escribiendo targets"):
        # end = min(start + chunk_size, num_samples)                
        tgt_chunk = load_tfrecord_to_numpy(test_file, metadata=metadata)['target']
        end = start + len(tgt_chunk)
        print(i, start, end)
        dset_target[start:end] = tgt_chunk
        start = end

In [ ]:
with h5py.File("test_mouse_copy.h5", "r+") as f:
    dset_target = f["target"]
    num_samples = dset_target.shape[0]

    start = 0
    for i, test_file in enumerate(test_files):
    # for start in tqdm(range(0, num_samples, chunk_size), desc="Escribiendo targets"):
        # end = min(start + chunk_size, num_samples)                
        tgt_chunk = load_tfrecord_to_numpy(test_file, metadata=metadata)['target']
        end = start + len(tgt_chunk)
        print(i, start, end)
        dset_target[start:end] = tgt_chunk
        start = end

In [ ]:
for i, (record_id, region_df) in enumerate(file_to_chunk.items()):
    # print(record_id)
    hashes_for_extracted = [ hash(x) for x in region_df.apply(extract_region, axis=1).to_list() ]
    sequences_for_record = [ v for k, v in get_sequences_for_record(record_id).items() ]
    hashes_for_fasta = [ hash(x) for x in sequences_for_record ]
    print(all([ hashes_for_extracted[i] == hashes_for_fasta[i] for i in range(256)]))

In [ ]:
file_to_chunk['valid-1-8'].apply(extract_region, axis=1)

In [ ]:
RECORD_ID = 'train-1-0'
sequences_from_tfr = get_sequences_for_record(RECORD_ID)
file_to_chunk = chunk_by_subset(mouse_seqs_df)
mouse_seqs_df.query("subset == 'train'").apply(extract_region, axis=1)

sequences_from_tfr

In [ ]:
SEQLEN = 131072
TO_LEFT, TO_RIGHT = SEQLEN, SEQLEN
WHICH_RECORD = 5

record_id = (record_ids := get_mouse_record_ids())[WHICH_RECORD]

print(f"{record_id=}")
sequences_from_tfr = get_sequences_for_record(record_id)
blast_results_df   = get_coords_from_blast(record_id)

expanded_regions_df = expand_regions(blast_results_df, to_left=1+TO_LEFT, to_right=0+TO_RIGHT)
sequences_from_ref  = expanded_regions_df.apply(extract_region, axis=1, expected_length=SEQLEN+TO_LEFT+TO_RIGHT)

# if this gives True, we are good
assert all([ v == sequences_from_ref.iloc[i][TO_LEFT:-TO_RIGHT] for i, (k, v) in enumerate(sequences_from_tfr.items()) ])

In [ ]:
subset = ['train', 'valid', 'test'][2]
mouse_seqs.query("subset == @subset")

In [ ]:
region = region_from_npy_filename("chr1_100033626_100426841.npy")
npy_file = (npy_files := os.listdir(MOUSE_NPY_FOLDER))[0]
one_sequence_from_before = onehot_to_seq(np.load(f"{MOUSE_NPY_FOLDER}/{npy_file}"))

In [ ]:
SHIFT = -3
one_sequence_from_before[SEQLEN+SHIFT:-SEQLEN+SHIFT] in mouse_seqs_set

In [ ]:
sequences.iloc[1] in mouse_seqs_set

In [ ]:
@interact
def show_sequence(start_index=widgets.IntSlider(min=1000000, max=60000000, step=100000)):
    print(chromosome_seq[start_index:(start_index+1000)])